# Feast Feature Store 따라해보기

이 노트북은 피처스토어(Feast)를 처음 접하는 분들을 위한 단계별 가이드입니다.

## 목표
- Feature Store 개념 이해
- Online 피처 조회 (실시간 예측용)
- Historical 피처 조회 (모델 학습용)
- Feature Service로 피처 번들링·조회

## 사전 준비

터미널에서 다음 명령을 실행했는지 확인하세요:

```bash
make up
make apply
make materialize
```

- `make up`: Redis + feast-dev 컨테이너 시작
- `make apply`: 피처 정의 적용
- `make materialize`: 소스(Parquet) 데이터를 Online Store(Redis)에 적재. Online 조회는 이 materialize된 데이터를 사용합니다.

## 1. FeatureStore 연결

In [47]:
from pathlib import Path
from feast import FeatureStore

# example_repo 경로 (컨테이너 내부: /workspace/example_repo)
repo_path = Path("/workspace/example_repo")
store = FeatureStore(repo_path=str(repo_path))

print("FeatureStore 연결 완료:", repo_path)

FeatureStore 연결 완료: /workspace/example_repo


## 2. 피처 정의 보기

등록된 feature view와 피처 스키마를 확인합니다. **정의 → 데이터 → 조회** 흐름을 이해하는 데 도움이 됩니다.

![](Lineage.png)

In [48]:
import pandas as pd
import sys

sys.path.insert(0, "/workspace/example_repo")
from entities import DEFAULT_DESCRIPTIONS

# Feast API로 등록된 feature view와 피처 목록 조회
rows = []
for fv in store.list_feature_views():
    for f in fv.schema:
        desc = f.description or DEFAULT_DESCRIPTIONS.get(f.name, "-")
        rows.append({
            "feature_view": fv.name,
            "entity": ", ".join(getattr(e, "name", str(e)) for e in fv.entities),
            "feature": f.name,
            "dtype": str(f.dtype),
            "description": desc,
        })
pd.DataFrame(rows)

,feature_view,entity,feature,dtype,description
0,customer_stats,customer,customer_id,Int64,"고객 ID (Customer identifier, entity join key)"
1,customer_stats,customer,avg_order_value_30d,Float32,"최근 30일 평균 주문 금액, 원 (Average order value in the..."
2,customer_stats,customer,days_since_last_purchase,Int64,마지막 구매 경과일 (Days since the customer's last pur...
3,customer_stats,customer,purchase_count_30d,Int64,최근 30일 구매 건수 (Purchase count in the last 30 days)
4,customer_profile,customer,customer_id,Int64,"고객 ID (Customer identifier, entity join key)"
5,customer_profile,customer,is_vip,Int64,VIP 멤버십 여부 (0 또는 1) (VIP membership flag (0 or...
6,customer_profile,customer,signup_days_ago,Int64,가입 경과일 (Days since customer signup)


## 3. 전체 데이터셋 보기

아래는 소스 Parquet의 전체 데이터입니다.

- **customer_stats**: 시계열 피처, **3명 × 10일 = 30행**
- **customer_profile**: 정적 피처, **3명 × 1행** (고객당 1행)

- **Online 조회**: 각 고객의 **가장 최근(03-10)** 행을 반환 (customer_stats) + 프로필 (customer_profile)
- **Historical 조회**: **지정한 시점(예: 03-05) 이전의 최신** 행을 반환

다음 섹션에서 이 값들이 어떻게 조회되는지 확인해 보세요.

In [49]:
import pandas as pd

# 소스 Parquet 직접 읽기 (3명 × 10일 = 30행)
df_full = pd.read_parquet("/workspace/example_repo/data/customer_stats.parquet")
df_full = df_full.sort_values(["customer_id", "event_timestamp"])
# event_timestamp를 날짜만 보이게 (가독성)
df_display = df_full.copy()
df_display["event_timestamp"] = df_display["event_timestamp"].dt.strftime("%Y-%m-%d")
df_display

,customer_id,event_timestamp,created_timestamp,purchase_count_30d,avg_order_value_30d,days_since_last_purchase
0,1001,2026-03-01,2026-03-01 00:00:00+00:00,8,42000.0,3
1,1001,2026-03-02,2026-03-02 00:00:00+00:00,8,41500.0,2
2,1001,2026-03-03,2026-03-03 00:00:00+00:00,9,43000.0,1
3,1001,2026-03-04,2026-03-04 00:00:00+00:00,9,42800.0,0
4,1001,2026-03-05,2026-03-05 00:00:00+00:00,9,42500.0,1
5,1001,2026-03-06,2026-03-06 00:00:00+00:00,9,42200.0,2
6,1001,2026-03-07,2026-03-07 00:00:00+00:00,10,43500.0,0
7,1001,2026-03-08,2026-03-08 00:00:00+00:00,10,43200.0,1
8,1001,2026-03-09,2026-03-09 00:00:00+00:00,10,43000.0,2
9,1001,2026-03-10,2026-03-10 00:00:00+00:00,11,43800.0,0


In [50]:
# customer_profile: 고객당 1행 (정적 피처)
df_profile = pd.read_parquet("/workspace/example_repo/data/customer_profile.parquet")
df_profile = df_profile.sort_values("customer_id")
df_profile_display = df_profile.copy()
df_profile_display["event_timestamp"] = df_profile_display["event_timestamp"].dt.strftime("%Y-%m-%d")
df_profile_display

,customer_id,event_timestamp,created_timestamp,signup_days_ago,is_vip
0,1001,2026-03-01,2026-03-01 00:00:00+00:00,90,1
1,1002,2026-03-01,2026-03-01 00:00:00+00:00,180,0
2,1003,2026-03-01,2026-03-01 00:00:00+00:00,30,0


## 4. Online 피처 조회

**Online Store**: 실시간 예측 시점에 저지연으로 피처를 조회합니다. 위 표에서 각 고객의 **마지막 행(03-10)**이 Online 조회 결과와 일치합니다.

### 4.1 단일 feature view

`customer_id`로 한 feature view(`customer_stats`)의 최신 피처 값들을 가져옵니다.

In [51]:
response = store.get_online_features(
    features=[
        "customer_stats:purchase_count_30d",
        "customer_stats:avg_order_value_30d",
        "customer_stats:days_since_last_purchase",
    ],
    entity_rows=[
        {"customer_id": 1001},
        {"customer_id": 1002},
        {"customer_id": 1003},
    ],
).to_dict()

print(response)

{'customer_id': [1001, 1002, 1003], 'avg_order_value_30d': [43800.0, 85600.0, 21400.0], 'purchase_count_30d': [11, 5, 2], 'days_since_last_purchase': [0, 7, 6]}


In [52]:
# DataFrame으로 보기 좋게 출력
import pandas as pd

df = pd.DataFrame(response)
# Online은 최신(03-10) 시점이므로 event_timestamp 추가, 컬럼 순서 통일
df["event_timestamp"] = pd.to_datetime("2026-03-10")
cols = ["customer_id", "event_timestamp", "purchase_count_30d", "avg_order_value_30d", "days_since_last_purchase"]
df = df[[c for c in cols if c in df.columns]]
df

,customer_id,event_timestamp,purchase_count_30d,avg_order_value_30d,days_since_last_purchase
0,1001,2026-03-10,11,43800.0,0
1,1002,2026-03-10,5,85600.0,7
2,1003,2026-03-10,2,21400.0,6


### 4.2 다중 feature view 조회

한 번의 요청으로 여러 feature view에서 피처를 함께 가져올 수 있습니다. `customer_stats`(시계열)와 `customer_profile`(정적)를 조합해 실시간 예측에 활용할 수 있습니다.

In [53]:
response_multi = store.get_online_features(
    features=[
        "customer_stats:purchase_count_30d",
        "customer_stats:avg_order_value_30d",
        "customer_stats:days_since_last_purchase",
        "customer_profile:signup_days_ago",
        "customer_profile:is_vip",
    ],
    entity_rows=[
        {"customer_id": 1001},
        {"customer_id": 1002},
        {"customer_id": 1003},
    ],
).to_dict()
print(response_multi)

{'customer_id': [1001, 1002, 1003], 'avg_order_value_30d': [43800.0, 85600.0, 21400.0], 'purchase_count_30d': [11, 5, 2], 'days_since_last_purchase': [0, 7, 6], 'is_vip': [1, 0, 0], 'signup_days_ago': [90, 180, 30]}


In [54]:
# 다중 FV 결과를 DataFrame으로
df_multi = pd.DataFrame(response_multi)
df_multi["event_timestamp"] = pd.to_datetime("2026-03-10")
cols = ["customer_id", "event_timestamp", "purchase_count_30d", "avg_order_value_30d", "days_since_last_purchase", "signup_days_ago", "is_vip"]
df_multi = df_multi[[c for c in cols if c in df_multi.columns]]
df_multi

,customer_id,event_timestamp,purchase_count_30d,avg_order_value_30d,days_since_last_purchase,signup_days_ago,is_vip
0,1001,2026-03-10,11,43800.0,0,90,1
1,1002,2026-03-10,5,85600.0,7,180,0
2,1003,2026-03-10,2,21400.0,6,30,0


## 5. Historical 피처 조회

**Historical features**: 특정 시점의 피처 값으로, 모델 학습 데이터셋을 만들 때 사용합니다.

`event_timestamp`를 지정해 **point-in-time correct**한 피처를 조회합니다. 위 전체 데이터셋 표에서 `2026-03-05` 행의 값과 일치하는지 확인해 보세요. Online(최신일 2026-03-10)과는 다른 과거 시점 값이 반환됩니다.

### 5.1 단일 feature view

먼저 `customer_stats` 하나의 feature view에서만 피처를 가져옵니다.

In [55]:
import pandas as pd

# 고정 데이터셋 기준: 2026-03-01 ~ 2026-03-10 (10일치)
# 2026-03-05 시점으로 조회 → Online(03-10 최신값)과 다른 과거 시점 값 반환
entity_df = pd.DataFrame(
    {
        "customer_id": [1001, 1002, 1003],
        "event_timestamp": pd.to_datetime(["2026-03-05T00:00:00Z"] * 3),
    }
)

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "customer_stats:purchase_count_30d",
        "customer_stats:avg_order_value_30d",
        "customer_stats:days_since_last_purchase",
    ],
).to_df()

training_df

,customer_id,event_timestamp,purchase_count_30d,avg_order_value_30d,days_since_last_purchase
0,1001,2026-03-05 00:00:00+00:00,9,42500.0,1
1,1002,2026-03-05 00:00:00+00:00,5,86800.0,1
2,1003,2026-03-05 00:00:00+00:00,2,22500.0,0


### 5.2 다중 feature view 조회

`customer_stats`와 `customer_profile`을 함께 요청하면, 학습 데이터셋 생성 시 여러 feature view에서 피처를 한 번에 조합할 수 있습니다.

In [56]:
# 같은 entity_df로 두 feature view의 피처를 함께 조회
training_df_multi = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "customer_stats:purchase_count_30d",
        "customer_stats:avg_order_value_30d",
        "customer_stats:days_since_last_purchase",
        "customer_profile:signup_days_ago",
        "customer_profile:is_vip",
    ],
).to_df()

training_df_multi

,customer_id,event_timestamp,purchase_count_30d,avg_order_value_30d,days_since_last_purchase,signup_days_ago,is_vip
0,1001,2026-03-05 00:00:00+00:00,9,42500.0,1,90,1
1,1002,2026-03-05 00:00:00+00:00,5,86800.0,1,180,0
2,1003,2026-03-05 00:00:00+00:00,2,22500.0,0,30,0


## 6. Feature Service로 피처 조회

**Feature Service**는 여러 feature view를 묶어 **이름 있는 피처 세트**로 정의합니다. 팀 협업·모델별 피처 관리에 유용하며, `features` 인자에 FeatureService 객체를 전달해 조회할 수 있습니다.

**feature_views.py** 에 정의된 내용
```python
# --- Feature Service: 피처 번들링 ---
# 여러 feature view를 묶어 이름 있는 피처 세트로 정의 (팀 협업, 모델별 피처 관리에 유용)
recommendation_fs = FeatureService(
    name="recommendation_features",
    features=[customer_stats_fv, customer_profile_fv],
    description="추천 모델용 피처 (Recommendation model features)",
)
```

### 6.1 Feature Service로 Online 피처 조회

In [61]:
# feature repo에서 recommendation_fs (FeatureService) import
import sys
sys.path.insert(0, "/workspace/example_repo")
from feature_views import recommendation_fs

# 다중 FV와 동일 피처, 다른 조회 방식
response_fs = store.get_online_features(
    features=recommendation_fs,
    entity_rows=[{"customer_id": 1001}, {"customer_id": 1002}, {"customer_id": 1003}],
).to_dict()

df_fs = pd.DataFrame(response_fs)
df_fs["event_timestamp"] = pd.to_datetime("2026-03-10")
cols = ["customer_id", "event_timestamp", "purchase_count_30d", "avg_order_value_30d", "days_since_last_purchase", "signup_days_ago", "is_vip"]
df_fs = df_fs[[c for c in cols if c in df_fs.columns]]
df_fs

,customer_id,event_timestamp,purchase_count_30d,avg_order_value_30d,days_since_last_purchase,signup_days_ago,is_vip
0,1001,2026-03-10,11,43800.0,0,90,1
1,1002,2026-03-10,5,85600.0,7,180,0
2,1003,2026-03-10,2,21400.0,6,30,0


### 6.2 Feature Service로 Historical 피처 조회

In [62]:
# 다중 FV와 동일 피처, 다른 조회 방식
training_df_fs = store.get_historical_features(
    entity_df=entity_df,
    features=recommendation_fs,
).to_df()

training_df_fs

,customer_id,event_timestamp,purchase_count_30d,avg_order_value_30d,days_since_last_purchase,signup_days_ago,is_vip
0,1001,2026-03-05 00:00:00+00:00,9,42500.0,1,90,1
1,1002,2026-03-05 00:00:00+00:00,5,86800.0,1,180,0
2,1003,2026-03-05 00:00:00+00:00,2,22500.0,0,30,0


## 정리

| 조회 방식 | 용도 | 특징 |
|----------|------|------|
| **Online** | 실시간 예측 | Redis에서 빠르게 조회, `entity_rows`로 즉시 요청 |
| **Historical** | 모델 학습 | point-in-time correct, `event_timestamp`로 시점 지정 |

- 피처스토어는 **같은 피처 정의**로 학습(offline)과 서빙(online)을 일관되게 지원합니다. 
- 단일 또는 다중 feature view에서 필요한 피처를 한 번의 요청으로 가져올 수 있습니다.
- **Feature Service**로 피처를 묶어 이름 있는 세트로 관리할 수 있습니다.
- `make ui` 실행 시 http://localhost:8888 에서 Feast Web UI로 feature view, entity를 시각적으로 탐색할 수 있습니다.